In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [2]:
train = pd.read_csv('C:/Projects/Housing-Price-Prediction/data/processed/train_encoded.csv')
val = pd.read_csv('C:/Projects/Housing-Price-Prediction/data/processed/val_encoded.csv')
test = pd.read_csv('C:/Projects/Housing-Price-Prediction/data/processed/test_encoded.csv')

In [3]:
non_features = ['zipcode', 'price']

X_train, y_train = train.drop(columns=non_features), train['price']
X_val,   y_val   = val.drop(columns=non_features),   val['price']
X_test,  y_test  = test.drop(columns=non_features),  test['price']

print(list(X_train.columns) == list(X_val.columns) == list(X_test.columns))

True


In [4]:
def score(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs(y_true - y_pred) / y_true) * 100
    r2   = r2_score(y_true, y_pred)
    return {'RMSE': rmse, 'MAE': mae, 'MAPE_%': mape, 'R2': r2}

LightGBM

In [5]:
lgbm = lgb.LGBMRegressor(random_state=42, n_jobs=-1)
lgbm.fit(X_train, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.057841 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7304
[LightGBM] [Info] Number of data points in the train set: 574484, number of used features: 38
[LightGBM] [Info] Start training from score 339998.369859


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [6]:
results = []
for name, X, y in [('train', X_train, y_train), ('val', X_val, y_val)]:
    row = {'model': 'LightGBM (default)', 'split': name}
    row.update(score(y, lgbm.predict(X)))
    results.append(row)

results_df = pd.DataFrame(results)
print(results_df.round(2).to_string(index=False))

             model split     RMSE      MAE  MAPE_%   R2
LightGBM (default) train 41898.45 24061.59    8.83 0.98
LightGBM (default)   val 72584.99 34715.33    8.00 0.96


In [7]:
pred_val = lgbm.predict(X_val)
print("mean actual:   ", round(y_val.mean()))
print("mean predicted:", round(pred_val.mean()))

mean actual:    449353
mean predicted: 447632


XGBoost

In [8]:
xgbr = xgb.XGBRegressor(random_state=42, n_jobs=-1)
xgbr.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets i

In [9]:
for name, X, y in [('train', X_train, y_train), ('val', X_val, y_val)]:
    print(name, {k: round(v, 2) for k, v in score(y, xgbr.predict(X)).items()})

train {'RMSE': np.float64(30487.89), 'MAE': 19039.89, 'MAPE_%': np.float64(7.0), 'R2': 0.99}
val {'RMSE': np.float64(72476.75), 'MAE': 34149.58, 'MAPE_%': np.float64(7.66), 'R2': 0.96}


In [10]:
pred_val = xgbr.predict(X_val)
print("mean actual:   ", round(y_val.mean()))
print("mean predicted:", round(float(pred_val.mean())))

mean actual:    449353
mean predicted: 445344


CatBoost Regressor

In [11]:
cat = CatBoostRegressor(random_seed=42, verbose=0)
cat.fit(X_train, y_train)

CatBoostRegressor(loss_function='RMSE', random_seed=42, verbose=0)

In [12]:
for name, X, y in [('train', X_train, y_train), ('val', X_val, y_val)]:
    print(name, {k: round(v, 2) for k, v in score(y, cat.predict(X)).items()})

train {'RMSE': np.float64(28221.81), 'MAE': 17713.73, 'MAPE_%': np.float64(6.63), 'R2': 0.99}
val {'RMSE': np.float64(65517.89), 'MAE': 31339.05, 'MAPE_%': np.float64(7.09), 'R2': 0.97}


In [13]:
pred_val = cat.predict(X_val)
print("mean actual:   ", round(y_val.mean()))
print("mean predicted:", round(float(pred_val.mean())))

mean actual:    449353
mean predicted: 447139


In [14]:
clean_train = pd.read_csv('C:/Projects/Housing-Price-Prediction/data/processed/train_clean.csv', usecols=['zipcode', 'price', 'median_sale_price'])
clean_val   = pd.read_csv('C:/Projects/Housing-Price-Prediction/data/processed/val_clean.csv',   usecols=['zipcode', 'price', 'median_sale_price'])

for name, clean, enc in [('train', clean_train, train), ('val', clean_val, val)]:
    same_len  = len(clean) == len(enc)
    same_zip  = same_len and (clean['zipcode'].values == enc['zipcode'].values).all()
    same_price = same_len and np.allclose(clean['price'].values, enc['price'].values)
    print(name, "| same length:", same_len, "| zipcodes match:", same_zip, "| prices match:", same_price)

train | same length: True | zipcodes match: True | prices match: True
val | same length: True | zipcodes match: True | prices match: True


In [15]:
X_train_sp = X_train.assign(median_sale_price=clean_train['median_sale_price'].values)
X_val_sp   = X_val.assign(median_sale_price=clean_val['median_sale_price'].values)

feature_sets = {
    'A baseline':            (X_train,                          X_val),
    'B without year':        (X_train.drop(columns='year'),     X_val.drop(columns='year')),
    'C + median_sale_price': (X_train_sp,                       X_val_sp),
}
for k, (a, b) in feature_sets.items():
    print(k, a.shape, b.shape)

A baseline (574484, 38) (147885, 38)
B without year (574484, 37) (147885, 37)
C + median_sale_price (574484, 39) (147885, 39)


In [16]:
def make_models():
    return {
        'LightGBM': lgb.LGBMRegressor(random_state=42, n_jobs=-1),
        'XGBoost':  xgb.XGBRegressor(random_state=42, n_jobs=-1),
        'CatBoost': CatBoostRegressor(random_seed=42, verbose=0),
    }

In [17]:
for fs_name, (Xtr, Xva) in feature_sets.items():
    for m_name, model in make_models().items():
        model.fit(Xtr, y_train)
        s = score(y_val, model.predict(Xva))
        print(f"{fs_name:24s} {m_name:9s} RMSE {s['RMSE']:>9,.0f} | MAE {s['MAE']:>8,.0f} | MAPE {s['MAPE_%']:.2f}% | R2 {s['R2']:.3f}")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.074668 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7304
[LightGBM] [Info] Number of data points in the train set: 574484, number of used features: 38
[LightGBM] [Info] Start training from score 339998.369859
A baseline               LightGBM  RMSE    72,585 | MAE   34,715 | MAPE 8.00% | R2 0.959
A baseline               XGBoost   RMSE    72,477 | MAE   34,150 | MAPE 7.66% | R2 0.959
A baseline               CatBoost  RMSE    65,518 | MAE   31,339 | MAPE 7.09% | R2 0.967
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.064695 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7295
[LightGBM] [Info] Number of d

In [18]:
cat_final = CatBoostRegressor(random_seed=42, verbose=0)
cat_final.fit(X_train_sp, y_train)

pred = cat_final.predict(X_val_sp)
print({k: round(v, 2) for k, v in score(y_val, pred).items()})

{'RMSE': np.float64(62701.59), 'MAE': 29522.59, 'MAPE_%': np.float64(6.77), 'R2': 0.97}


In [19]:
err = pd.DataFrame({
    'zipcode':   val['zipcode'].values,
    'year':      val['year'].values,
    'month':     val['month'].values,
    'actual':    y_val.values,
    'predicted': pred,
    'sale_price': X_val_sp['median_sale_price'].values,
    'list_price': X_val_sp['median_list_price'].values,
    'homes_sold': X_val_sp['homes_sold'].values,
    'census_missing': X_val_sp['census_missing'].values,
})
err['error']     = err['predicted'] - err['actual']      # positive = overpredicted
err['abs_error'] = err['error'].abs()
err['pct_error'] = err['abs_error'] / err['actual'] * 100
print(err.shape)

(147885, 12)


In [20]:
sq = err['error'] ** 2
for frac in [0.001, 0.01, 0.05]:
    top = sq.nlargest(int(len(err) * frac)).sum() / sq.sum()
    print(f"worst {frac:.1%} of rows -> {top:.1%} of total squared error")

print("\nmedian abs error:", round(err['abs_error'].median()))
print("mean abs error:  ", round(err['abs_error'].mean()))

worst 0.1% of rows -> 27.6% of total squared error
worst 1.0% of rows -> 59.4% of total squared error
worst 5.0% of rows -> 80.3% of total squared error

median abs error: 15726
mean abs error:   29523


In [21]:
print(err.nlargest(20, 'abs_error').round(0).to_string(index=False))

 zipcode  year  month    actual  predicted  sale_price  list_price  homes_sold  census_missing      error  abs_error  pct_error
   90210  2021      3 4716742.0  2442463.0   4000000.0   5575000.0        83.0               0 -2274280.0  2274280.0       48.0
   90210  2021      2 4732641.0  2476921.0   5000000.0   5970000.0        67.0               0 -2255720.0  2255720.0       48.0
    2199  2021      2 3739642.0  1839621.0   5900000.0   1500000.0         1.0               0 -1900021.0  1900021.0       51.0
    2199  2021      1 3737683.0  1839621.0   5900000.0   1500000.0         1.0               0 -1898062.0  1898062.0       51.0
   11962  2021     11 4715595.0  2923663.0   2050000.0   1800000.0         1.0               0 -1791932.0  1791932.0       38.0
   94027  2021     12 7705425.0  5930732.0   7500000.0   9369440.0        32.0               0 -1774692.0  1774692.0       23.0
   11962  2021     12 4827316.0  3111527.0   3200000.0   1800000.0         2.0               0 -1715789.

In [22]:
by_zip = (err.groupby('zipcode')
             .agg(rows=('error', 'size'),
                  mean_abs_error=('abs_error', 'mean'),
                  total_sq_error=('error', lambda s: (s**2).sum()))
             .sort_values('total_sq_error', ascending=False))
by_zip['share_of_sq_error'] = by_zip['total_sq_error'] / by_zip['total_sq_error'].sum()
print("zipcodes in val:", len(by_zip))
print(by_zip.head(10).round(3).to_string())
print("top 10 zipcodes share:", round(by_zip['share_of_sq_error'].head(10).sum(), 3))

zipcodes in val: 6198
         rows  mean_abs_error  total_sq_error  share_of_sq_error
zipcode                                                         
94027      24     1069534.754    2.994657e+13              0.052
2199       24      863764.951    2.515937e+13              0.043
90210      24      590577.305    1.753377e+13              0.030
11962      19      585006.833    1.404259e+13              0.024
10001      24      655687.621    1.232313e+13              0.021
92067      24      546567.019    1.158292e+13              0.020
7723       24      622692.362    9.852795e+12              0.017
11976      24      476865.118    9.051873e+12              0.016
11932      23      522003.124    9.043488e+12              0.016
94970      24      504826.528    8.715934e+12              0.015
top 10 zipcodes share: 0.253


In [23]:
err['price_band'] = pd.qcut(err['actual'], 5, labels=['lowest', 'low', 'mid', 'high', 'highest'])
err['thin_market'] = np.where(err['homes_sold'] <= 2, 'homes_sold <= 2', 'homes_sold > 2')

for col in ['price_band', 'thin_market', 'census_missing', 'year']:
    g = err.groupby(col, observed=True).agg(rows=('error', 'size'),
                                            MAE=('abs_error', 'mean'),
                                            mean_pct_error=('pct_error', 'mean'),
                                            bias=('error', 'mean'))
    print(g.round(1).to_string(), "\n")

             rows      MAE  mean_pct_error    bias
price_band                                        
lowest      29577  13619.0             9.8  4156.6
low         29577  16370.2             6.2   638.3
mid         29577  20144.6             5.6   732.7
high        29577  27029.2             5.4 -3301.8
highest     29577  70450.0             6.8 -6593.1 

                   rows      MAE  mean_pct_error    bias
thin_market                                             
homes_sold <= 2    7476  54077.6            16.6 -1511.9
homes_sold > 2   140409  28215.2             6.2  -839.5 

                  rows      MAE  mean_pct_error    bias
census_missing                                         
0               136633  29009.7             6.7 -1363.2
1                11252  35750.2             7.6  5073.0 

       rows      MAE  mean_pct_error    bias
year                                        
2020  73856  24687.5             6.5  7510.5
2021  74029  34346.4             7.1 -9237.8 



Feature importance on cat_final

Built-in importance (how much a column was used to fit train)

In [24]:
imp = pd.Series(cat_final.get_feature_importance(), index=X_train_sp.columns)
imp = imp.sort_values(ascending=False)
print(imp.round(2).to_string())

median_sale_price          28.78
median_list_price          27.29
Median Home Value          11.78
Per Capita Income           4.33
lng                         3.05
median_list_ppsf            2.24
restaurant                  2.16
median_ppsf                 2.03
city_encoded                1.94
Median Age                  1.64
Total Housing Units         1.55
year                        1.30
school                      1.03
park                        1.01
station                     0.99
Median Rent                 0.98
Total Labor Force           0.97
Total Population            0.73
bank                        0.71
lat                         0.66
commute_count               0.61
pending_sales               0.58
inventory                   0.55
hospital                    0.52
poverty_universe_total      0.50
Unemployed Population       0.32
new_listings                0.31
supermarket                 0.29
avg_sale_to_list            0.21
homes_sold                  0.19
mall      

Importance by cluster

In [25]:
clusters = {
    'price level': ['median_sale_price', 'median_list_price', 'median_ppsf',
                    'median_list_ppsf', 'Median Home Value'],
    'zipcode size (census)': ['Total Population', 'school_universe_total', 'poverty_universe_total',
                              'Total Labor Force', 'commute_count', 'Total Housing Units',
                              'Unemployed Population'],
    'market activity': ['homes_sold', 'new_listings', 'pending_sales', 'inventory'],
    'amenities': ['bank', 'restaurant', 'supermarket', 'park', 'school', 'hospital', 'bus', 'mall', 'station'],
    'location': ['lat', 'lng', 'city_encoded'],
    'time': ['year', 'quarter', 'month'],
}
for k, cols in clusters.items():
    print(f"{k:24s} {imp[imp.index.isin(cols)].sum():6.1f}")
print("not in any cluster:", [c for c in imp.index if not any(c in v for v in clusters.values())])

price level                72.1
zipcode size (census)       4.9
market activity             1.6
amenities                   7.0
location                    5.7
time                        1.4
not in any cluster: ['Per Capita Income', 'Median Age', 'Median Rent', 'avg_sale_to_list', 'median_dom', 'sold_above_list', 'off_market_in_two_weeks', 'census_missing']


Permutation importance on val (how much val RMSE worsens without it)

In [26]:
from sklearn.inspection import permutation_importance

idx = X_val_sp.sample(25000, random_state=42).index
r = permutation_importance(cat_final, X_val_sp, y_val,
                           scoring='neg_root_mean_squared_error',
                           n_repeats=3, random_state=42, n_jobs=1)

perm = pd.Series(r.importances_mean, index=X_val_sp.columns).sort_values(ascending=False)
print(perm.round(0).to_string())

Median Home Value          135510.0
median_list_price          100415.0
median_sale_price           79841.0
Per Capita Income           53592.0
Total Housing Units         19660.0
median_ppsf                 13479.0
lng                         13400.0
median_list_ppsf            13364.0
city_encoded                12112.0
Total Labor Force            8478.0
Median Rent                  7516.0
restaurant                   7500.0
Total Population             6811.0
Median Age                   6704.0
commute_count                5563.0
school                       5277.0
bank                         4363.0
park                         3333.0
station                      2160.0
poverty_universe_total       2033.0
Unemployed Population        1785.0
pending_sales                1759.0
school_universe_total        1527.0
inventory                    1461.0
supermarket                  1281.0
new_listings                 1117.0
hospital                      925.0
lat                         

Target experiments on CatBoost

Check the reference column is usable

In [27]:
ref = 'median_sale_price'
for name, X in [('train', X_train_sp), ('val', X_val_sp)]:
    r = X[ref]
    print(name, "| zeros:", (r == 0).sum(), "| NaN:", r.isna().sum(), "| min:", r.min())

train | zeros: 0 | NaN: 0 | min: 600.0
val | zeros: 0 | NaN: 0 | min: 4750.0


Experiment 1, log of price (the control)

In [28]:
m1 = CatBoostRegressor(random_seed=42, verbose=0)
m1.fit(X_train_sp, np.log(y_train))

pred1 = np.exp(m1.predict(X_val_sp))
print("log(price):", {k: round(v, 2) for k, v in score(y_val, pred1).items()})

log(price): {'RMSE': np.float64(68967.26), 'MAE': 29558.19, 'MAPE_%': np.float64(6.39), 'R2': 0.96}


Experiment 2, log ratio against median_sale_price

In [29]:
ref_tr, ref_va = X_train_sp[ref], X_val_sp[ref]

m2 = CatBoostRegressor(random_seed=42, verbose=0)
m2.fit(X_train_sp, np.log(y_train / ref_tr))

pred2 = np.exp(m2.predict(X_val_sp)) * ref_va
print("log(price/sale):", {k: round(v, 2) for k, v in score(y_val, pred2).items()})

log(price/sale): {'RMSE': np.float64(94965.42), 'MAE': 29848.06, 'MAPE_%': np.float64(6.36), 'R2': 0.93}


Side by side

In [30]:
def slices(pred, label):
    e = pd.DataFrame({'actual': y_val.values, 'pred': pred,
                      'zipcode': val['zipcode'].values, 'year': val['year'].values,
                      'homes_sold': X_val_sp['homes_sold'].values})
    e['err'] = e['pred'] - e['actual']
    e['band'] = pd.qcut(e['actual'], 5, labels=['lowest','low','mid','high','highest'])
    out = {
        'label': label,
        'bias highest band': round(e.loc[e['band']=='highest','err'].mean()),
        'bias 2021':         round(e.loc[e['year']==2021,'err'].mean()),
        'MAE thin (<=2)':    round(e.loc[e['homes_sold']<=2,'err'].abs().mean()),
        'MAE 94027':         round(e.loc[e['zipcode']==94027,'err'].abs().mean()),
        'MAE 90210':         round(e.loc[e['zipcode']==90210,'err'].abs().mean()),
    }
    return out

rows = [slices(pred, 'baseline (price)'), slices(pred1, 'log(price)'), slices(pred2, 'log(price/sale)')]
print(pd.DataFrame(rows).to_string(index=False))

           label  bias highest band  bias 2021  MAE thin (<=2)  MAE 94027  MAE 90210
baseline (price)              -6593      -9238           54078    1069535     590577
      log(price)             -11872     -11633           55549    1326234    1429803
 log(price/sale)                122      -7419           65911     728324    1036321


In [31]:
train_clean = pd.read_csv('C:/Projects/Housing-Price-Prediction/data/processed/train_clean.csv')
val_clean = pd.read_csv('C:/Projects/Housing-Price-Prediction/data/processed/val_clean.csv')
test_clean = pd.read_csv('C:/Projects/Housing-Price-Prediction/data/processed/test_clean.csv')

In [32]:
train[['date', 'city', 'median_sale_price']] = train_clean[['date',  'city', 'median_sale_price']]

val[['date',  'city', 'median_sale_price']] = val_clean[['date',  'city', 'median_sale_price']]

test[['date',  'city', 'median_sale_price']] = test_clean[['date',  'city', 'median_sale_price']]

In [33]:
train.to_csv(f'C:/Projects/Housing-Price-Prediction/data/processed/train_final.csv', index=False)
val.to_csv('C:/Projects/Housing-Price-Prediction/data/processed/val_final.csv', index=False)
test.to_csv('C:/Projects/Housing-Price-Prediction/data/processed/test_final.csv', index=False)